In [1]:
from model_setup_colorado import X, y, groups, evaluate

Rows: 343 | Schools: 22
Share labeled 1: 0.417


Cell 2: baseline

In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

results = {}
results["Dummy"] = evaluate(DummyClassifier(strategy="most_frequent"), "Dummy")

logit = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(max_iter=1000))
results["Logistic Regression"] = evaluate(logit, "Logistic Regression")

Dummy {'accuracy': np.float64(0.583), 'precision': np.float64(0.0), 'recall': np.float64(0.0), 'f1': np.float64(0.0), 'roc_auc': np.float64(0.5)}
Logistic Regression {'accuracy': np.float64(0.501), 'precision': np.float64(0.417), 'recall': np.float64(0.453), 'f1': np.float64(0.385), 'roc_auc': np.float64(0.551)}


/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/lib/python3.13/site-packages/s

Cell 3: tree models

In [3]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

results["Random Forest"] = evaluate(make_pipeline(SimpleImputer(strategy="median"),
    RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)),
    "Random Forest")

results["XGBoost"] = evaluate(XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
    subsample=0.8, random_state=42, eval_metric="logloss"), "XGBoost")

results["HistGradientBoosting"] = evaluate(HistGradientBoostingClassifier(max_depth=3,
    learning_rate=0.05, min_samples_leaf=10, random_state=42), "HistGradientBoosting")

Random Forest {'accuracy': np.float64(0.662), 'precision': np.float64(0.648), 'recall': np.float64(0.514), 'f1': np.float64(0.543), 'roc_auc': np.float64(0.731)}
XGBoost {'accuracy': np.float64(0.72), 'precision': np.float64(0.679), 'recall': np.float64(0.651), 'f1': np.float64(0.655), 'roc_auc': np.float64(0.768)}
HistGradientBoosting {'accuracy': np.float64(0.714), 'precision': np.float64(0.685), 'recall': np.float64(0.6), 'f1': np.float64(0.634), 'roc_auc': np.float64(0.765)}


Cell 4: table and save

In [4]:
import pandas as pd
table = pd.DataFrame(results).T
display(table)
table.to_csv("results_colorado.csv")

,accuracy,precision,recall,f1,roc_auc
Dummy,0.583,0.000,0.000,0.000,0.500
Logistic Regression,0.501,0.417,0.453,0.385,0.551
Random Forest,0.662,0.648,0.514,0.543,0.731
XGBoost,0.720,0.679,0.651,0.655,0.768
HistGradientBoosting,0.714,0.685,0.600,0.634,0.765


Cell 5: how noisy are the scores?

In [5]:
from sklearn.model_selection import cross_validate, GroupKFold
res = cross_validate(logit, X, y, cv=GroupKFold(5), groups=groups, scoring=["roc_auc", "f1"])
print("ROC-AUC per fold:", res["test_roc_auc"].round(2))
print("F1 per fold:", res["test_f1"].round(2))

ROC-AUC per fold: [0.56 0.52 0.54 0.64 0.5 ]
F1 per fold: [0.53 0.26 0.18 0.51 0.44]


Check

In [6]:
from sklearn.model_selection import cross_validate, GroupKFold
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                    subsample=0.8, random_state=42, eval_metric="logloss")

res = cross_validate(xgb, X, y, cv=GroupKFold(5), groups=groups, scoring=["roc_auc", "f1"])
print("XGBoost ROC-AUC per fold:", res["test_roc_auc"].round(2))
print("XGBoost F1 per fold:", res["test_f1"].round(2))

def run(cols, name):
    r = cross_validate(xgb, X[cols], y, cv=GroupKFold(5), groups=groups, scoring=["roc_auc"])
    print(name, "ROC-AUC:", r["test_roc_auc"].mean().round(3))

run(["CIP_FAMILY"], "subject only")
run([c for c in X.columns if c != "CIP_FAMILY"], "everything except subject")

XGBoost ROC-AUC per fold: [0.77 0.78 0.75 0.85 0.7 ]
XGBoost F1 per fold: [0.66 0.72 0.65 0.69 0.55]
subject only ROC-AUC: 0.793
everything except subject ROC-AUC: 0.563


Check 2

In [7]:
from model_setup_colorado import df, X, y, groups
import numpy as np
from sklearn.model_selection import cross_validate, GroupKFold
from xgboost import XGBClassifier

t = (df.assign(LABEL=y)
       .groupby("CIP_FAMILY")
       .agg(programs=("LABEL", "size"), share_beating=("LABEL", "mean"), example_program=("CIPDESC", "first"))
       .sort_values("share_beating", ascending=False).round(2))
display(t)

X2 = X.copy()
X2["A_MEDIAN"] = df["A_MEDIAN"]
X2["LOC_QUOTIENT"] = df["LOC_QUOTIENT"]
X2["TOT_EMP_LOG"] = np.log1p(df["TOT_EMP"])

xgb = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                    subsample=0.8, random_state=42, eval_metric="logloss")
r = cross_validate(xgb, X2, y, cv=GroupKFold(5), groups=groups, scoring=["roc_auc", "f1"])
print("with local wage columns: ROC-AUC", r["test_roc_auc"].mean().round(3), "| F1", r["test_f1"].mean().round(3))

,programs,share_beating,example_program
CIP_FAMILY,,,
39,1,1.00,Bible/Biblical Studies.
16,1,1.00,"Linguistic, Comparative, and Related Language ..."
11,16,0.94,"Computer and Information Sciences, General."
43,23,0.87,Criminal Justice and Corrections.
22,6,0.83,Legal Support Services.
15,21,0.76,Environmental Control Technologies/Technicians.
46,20,0.70,Electrical and Power Transmission Installers.
52,33,0.64,Accounting and Related Services.
45,2,0.50,Geography and Cartography.


with local wage columns: ROC-AUC 0.781 | F1 0.656
